# nb02 — Animação de satélite GOES-16 (todos os 16 canais)

Escolha **onda + canal + passo**; o notebook baixa (com cache local em `cache_abi/`), monta a
**animação com player** (play/pause/scrub) e pode sobrepor os **raios (GLM)**. Dá para exportar GIF.

> Roda na sua máquina (precisa de internet). Primeira vez de cada canal/janela baixa; depois usa o cache.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.animation as manim
import cartopy.crs as ccrs
import ipywidgets as W; from IPython.display import display, HTML
import importlib, ondas_config, viz_helpers as vh
importlib.reload(ondas_config); importlib.reload(vh)  # pega a versão mais nova sem reiniciar o kernel
from ondas_config import ONDAS, CAIXA
plt.rcParams['animation.embed_limit']=100  # MB p/ jshtml
vh.lista_bandas()

## Controles
O rótulo do canal mostra o comprimento de onda e o tipo (refl = reflectância diurna; emis = temperatura de brilho, 24 h).

In [ ]:
op_onda=list(ONDAS)
op_banda=[(f'C{b:02d} — {vh.BANDAS[b][0]} ({vh.BANDAS[b][1]}µm, {vh.BANDAS[b][2]})', b) for b in range(1,17)]
w_onda =W.Dropdown(options=op_onda, value=[o for o in op_onda if o.startswith('O1')][0], description='onda')
w_banda=W.Dropdown(options=op_banda, value=13, description='canal')
w_passo=W.IntSlider(value=1,min=1,max=6,description='passo (h)')
w_h0=W.IntSlider(value=11,min=0,max=23,description='hora ini (UT)')
w_h1=W.IntSlider(value=23,min=1,max=24,description='hora fim (UT)')
w_glm=W.Checkbox(value=True, description='sobrepor raios (GLM)')
display(W.VBox([w_onda,w_banda,W.HBox([w_h0,w_h1,w_passo]),w_glm]))

## Baixar os quadros (usa cache)

In [ ]:
fs=vh._fs()
whens=vh.horarios(w_onda.value, passo_h=w_passo.value, h0=w_h0.value, h1=w_h1.value)
print(f'{len(whens)} quadros pedidos para {w_onda.value}, canal C{w_banda.value:02d}')
pares=vh.baixa_abi(w_banda.value, whens, fs=fs)
frames=[]
for w,p in pares:
    campo=vh.le_abi(p, w_banda.value, CAIXA)
    if campo is None: continue
    campo['when']=campo['when'] or w
    if w_glm.value:
        fla,flo=vh.carrega_glm(w, CAIXA, jan_min=max(2,w_passo.value*3), fs=fs)
        campo['glm']=(fla,flo)
    frames.append(campo)
print('quadros prontos:',len(frames))

## Montar a animação (player interativo)

In [ ]:
assert frames, 'nenhum quadro — rode a célula anterior'
dado0,vmin,vmax,cmap,rot=vh.escala(frames[0])
fig,ax=vh.novo_mapa(CAIXA, figsize=(7.5,7.5))
im=ax.pcolormesh(frames[0]['lon'],frames[0]['lat'],dado0,cmap=cmap,vmin=vmin,vmax=vmax,
                 shading='auto',transform=ccrs.PlateCarree(),zorder=1)
cb=fig.colorbar(im,ax=ax,fraction=0.04,label=rot)
sc=ax.scatter([],[],s=3,c='#E69F00',alpha=.6,transform=ccrs.PlateCarree(),zorder=5)
vh.extensao(ax,CAIXA)
tit=ax.set_title('')
def upd(i):
    f=frames[i]; d,_,_,_,_=vh.escala(f)
    im.set_array(d.ravel())
    if 'glm' in f and f['glm'][0].size: sc.set_offsets(np.c_[f['glm'][1],f['glm'][0]])
    else: sc.set_offsets(np.empty((0,2)))
    tit.set_text(f"C{w_banda.value:02d} {vh.BANDAS[w_banda.value][0]} — {f['when']:%Y-%m-%d %H:%MUT}")
    return im,sc,tit
ani=manim.FuncAnimation(fig,upd,frames=len(frames),interval=350,blit=False)
plt.close(fig)
HTML(ani.to_jshtml())

## Exportar GIF (opcional)

In [ ]:
nome=f'anim_{w_onda.value}_C{w_banda.value:02d}.gif'
ani.save(nome, writer=manim.PillowWriter(fps=3))
print('salvo:',nome)

## Scrubber de quadro único (leve — sem montar vídeo)
Alternativa rápida: arraste o slider para ver quadro a quadro.

In [ ]:
def ver(i=0):
    f=frames[i]; d,vmin,vmax,cmap,rot=vh.escala(f)
    fig,ax=vh.novo_mapa(CAIXA, figsize=(7,7))
    m=ax.pcolormesh(f['lon'],f['lat'],d,cmap=cmap,vmin=vmin,vmax=vmax,
                    shading='auto',transform=ccrs.PlateCarree(),zorder=1)
    fig.colorbar(m,ax=ax,label=rot)
    if 'glm' in f and f['glm'][0].size:
        ax.scatter(f['glm'][1],f['glm'][0],s=3,c='#E69F00',alpha=.6,
                   transform=ccrs.PlateCarree(),zorder=5)
    vh.extensao(ax,CAIXA)
    ax.set_title(f"{f['when']:%Y-%m-%d %H:%MUT}"); plt.show()
W.interact(ver, i=W.IntSlider(min=0,max=len(frames)-1,step=1,value=0,description='quadro'));